# Chess-Coach Agent — LoRA on Kaggle T4

Trains a **Gemma 4 E2B** LoRA adapter (bf16, single T4) on the v1.2 skill-routing SFT corpus, then zips the adapter for download. You serve it locally as q4_0 GGUF on your RTX 4060.

## Why E2B bf16 (not E4B QLoRA)
E4B in 4-bit shards across both T4s and OOMs GPU 1 at the cross-entropy over Gemma's ~262k vocab, even at seq 1280. **E2B in bf16 fits one T4** with the multimodal towers offloaded to CPU. The base is the QAT (`qat-q4_0-unquantized`) checkpoint — stored in **full precision** (NOT weak); QAT just makes it robust to the q4_0 serving quant you apply later on the 4060.

## Kaggle setup (do this first)
1. **Settings → Accelerator → GPU T4 x2** (single T4 is enough; x2 is fine — `NO_4BIT` keeps it on one card).
2. **Settings → Internet → On** (clone repo + download base model).
3. **Add-ons → Secrets** → add `HF_TOKEN` (HF token, read scope). Accept the Gemma license once: https://huggingface.co/google/gemma-4-E2B-it-qat-q4_0-unquantized
4. Run cells **top to bottom**. Time ≈ base-download (~6 min) + training (depends on `MAX_STEPS`).

## Corpus
Cell 6 reads the regenerated, grounded v1.2 corpus (stored gzipped on the branch). If a cell reports `rows= 0`, the branch is missing the corpus — pull again.

## If it still OOMs
Lower `MAX_SEQ` in Cell 1 (1280 → 1024 → 768). Manifest-heavy rows get truncated below ~1024, so prefer 1024 before 768.

In [ ]:
# Cell 1 — config
import os

MODEL = "gemma4_e2b"  # E2B bf16 fits ONE T4. E4B QLoRA OOMs on 2x T4 — do not use here.
HF_REPO = {
    "gemma4_e4b": "google/gemma-4-E4B-it-qat-q4_0-unquantized",
    "gemma4_e2b": "google/gemma-4-E2B-it-qat-q4_0-unquantized",
}[MODEL]

# bf16 (unquantized) single-GPU. The QAT base is full-precision (NOT weak) and
# stays robust to the q4_0 serving quant you do later on the 4060. Setting this
# False runs 4-bit, which shards across both T4s and OOMs GPU 1.
NO_4BIT = True

CHESS_REPO_URL = "https://github.com/RyanDev1st/llm-and-engine.git"
CHESS_BRANCH = "feat/chess-coach-sft"
WORKDIR = "/kaggle/working/llm-and-engine"
OUTPUT = "gemma4_chess_kaggle"

# training knobs (bf16 LoRA, single-T4-friendly)
MAX_STEPS = 800
RANK = 8
TARGETS = "qv"
GRAD_ACCUM = 8
MAX_SEQ = 1280
print("config:", MODEL, HF_REPO, "no_4bit=", NO_4BIT)


In [ ]:
# Cell 2 — GPU preflight (stop early if no GPU)
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
                      "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU — set Accelerator to GPU T4 x2"
print("cuda", torch.version.cuda, "| device", torch.cuda.get_device_name(0),
      "| count", torch.cuda.device_count())


In [ ]:
# Cell 3 — clone repo (skip LFS weights; we only need code + data)
import os, subprocess

def run(cmd, **kw):
    print(">", " ".join(cmd)); return subprocess.run(cmd, check=True, text=True, **kw)

env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
if not os.path.exists(WORKDIR):
    run(["git", "clone", "--depth", "1", "--branch", CHESS_BRANCH, CHESS_REPO_URL, WORKDIR], env=env)
else:
    run(["git", "-C", WORKDIR, "pull", "--ff-only"], env=env)
run(["git", "-C", WORKDIR, "log", "-1", "--oneline"])
os.environ["PYTHONPATH"] = f"{WORKDIR}/src/llm"
print("PYTHONPATH=", os.environ["PYTHONPATH"])


In [ ]:
# Cell 4 — deps (Gemma 4 needs a recent transformers)
import subprocess, sys
def pip(*a):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)
pip("-U", "transformers>=5.8", "accelerate>=1.2", "peft>=0.14", "trl>=0.13",
    "bitsandbytes>=0.45", "datasets", "sentencepiece", "protobuf", "python-chess")
import transformers, peft, bitsandbytes
print("transformers", transformers.__version__, "| peft", peft.__version__,
      "| bnb", bitsandbytes.__version__)


In [ ]:
# Cell 5 — HF login + download base model (token from Kaggle Secrets)
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, snapshot_download

login(UserSecretsClient().get_secret("HF_TOKEN"))
dest = f"{WORKDIR}/src/llm/models/{MODEL}"
snapshot_download(repo_id=HF_REPO, local_dir=dest,
                  allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt", "tokenizer*"])
print("base model at", dest)
print(sorted(os.listdir(dest)))


In [ ]:
# Cell 6 - data check (must be the REGENERATED grounded corpus; stored gzipped)
import os, gzip
for name in ("v1_2_train.jsonl", "v1_2_val.jsonl"):
    base = f"{WORKDIR}/data/sft/{name}"
    path = base if os.path.exists(base) else base + ".gz"
    if not os.path.exists(path):
        n = 0
    elif path.endswith(".gz"):
        with gzip.open(path, "rt", encoding="utf-8") as fh:
            n = sum(1 for _ in fh)
    else:
        n = sum(1 for _ in open(path, encoding="utf-8"))
    print(name, "rows=", n, "(", os.path.basename(path), ")")
    assert n > 0, f"missing/empty {path} - commit the regenerated corpus to the branch first"


In [ ]:
# Cell 7 — train bf16 LoRA -> adapter under runs/<OUTPUT>
import subprocess, sys, os
cmd = [sys.executable, "-m", "llm_training.run_train",
       "--model", MODEL, "--max-steps", str(MAX_STEPS), "--rank", str(RANK),
       "--targets", TARGETS, "--grad-accum", str(GRAD_ACCUM), "--max-seq", str(MAX_SEQ),
       "--output", OUTPUT]
if NO_4BIT:
    cmd.append("--no-4bit")  # bf16 single-GPU; without this it runs 4-bit + shards + OOMs
print(">", " ".join(cmd))
subprocess.run(cmd, check=True, cwd=WORKDIR,
               env={**os.environ, "PYTHONPATH": f"{WORKDIR}/src/llm"})


In [ ]:
# Cell 8 — zip the adapter to /kaggle/working for download
import shutil, os
run_dir = f"{WORKDIR}/runs/{OUTPUT}"
print("adapter files:", os.listdir(run_dir))
out_zip = f"/kaggle/working/{OUTPUT}_adapter"
shutil.make_archive(out_zip, "zip", run_dir)
print("download from the Output panel:", out_zip + ".zip")
